# Feature Engineering for Advanced Models
Load full year data and create comprehensive features for Linear and XGBoost model

In [56]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Utility Functions

In [57]:
def get_black_friday(year):
    """
    Returns the Black Friday date for a given year
    Black Friday = The day after the 4th Thursday in November (Thanksgiving)
    """
    nov_first = datetime(year, 11, 1)
    days_until_thursday = (3 - nov_first.weekday()) % 7
    first_thursday = nov_first + timedelta(days=days_until_thursday)
    thanksgiving = first_thursday + timedelta(weeks=3)
    black_friday = thanksgiving + timedelta(days=1)
    return black_friday

def get_us_holidays(year):
    """
    Returns major US holidays for a given year
    """
    holidays = [
        datetime(year, 1, 1),   # New Year's Day
        datetime(year, 7, 4),   # Independence Day
        datetime(year, 12, 25), # Christmas
    ]
    
    # Thanksgiving (4th Thursday of November)
    thanksgiving = get_black_friday(year) - timedelta(days=1)
    holidays.append(thanksgiving)
    
    # Black Friday
    holidays.append(get_black_friday(year))
    
    return holidays

# Test functions
print("Black Friday 2024:", get_black_friday(2024))
print("US Holidays 2024:", get_us_holidays(2024))

Black Friday 2024: 2024-11-29 00:00:00
US Holidays 2024: [datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 7, 4, 0, 0), datetime.datetime(2024, 12, 25, 0, 0), datetime.datetime(2024, 11, 28, 0, 0), datetime.datetime(2024, 11, 29, 0, 0)]


## Load Full Year Data

In [58]:
# Load full year data for each year
data_dir = '../dataset/raw/'
all_years_data = []

csv_files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))

# Load all PJM CSV files
if csv_files:
    print("Loading PJM data...")
    
    all_pjm_data = []
    
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        try:
            df = pd.read_csv(file_path)
            df['datetime_beginning_ept'] = pd.to_datetime(df['datetime_beginning_ept'])
            all_pjm_data.append(df)
            print(f"  ✓ {file_name}: {len(df):,} rows")
        except Exception as e:
            print(f"  ✗ Error loading {file_name}: {e}")
    
    # Combine all data
    full_df = pd.concat(all_pjm_data, ignore_index=True)
    
    print(f"\n{'='*80}")
    print("PJM Data Summary:")
    print(f"{'='*80}")
    print(f"Total rows: {len(full_df):,}")
    print(f"Total columns: {len(full_df.columns)}")
    print(f"Date range: {full_df['datetime_beginning_ept'].min()} to {full_df['datetime_beginning_ept'].max()}")
    print(f"Load areas: {full_df['load_area'].nunique()}")
    
    print(f"\nColumns: {full_df.columns.tolist()}")
    
    print(f"\nFirst 5 rows:")
    display(full_df.head())
    
    # Store date range for weather data
    pjm_start_date = full_df['datetime_beginning_ept'].min()
    pjm_end_date = full_df['datetime_beginning_ept'].max()
    
    print(f"\n✓ PJM data loaded successfully")
else:
    full_df = None
    print("\n✗ No PJM data available")

Loading PJM data...
  ✓ hrl_load_metered_2012.csv: 193,248 rows
  ✓ hrl_load_metered_2013.csv: 197,857 rows
  ✓ hrl_load_metered_2014.csv: 201,480 rows
  ✓ hrl_load_metered_2015.csv: 227,165 rows
  ✓ hrl_load_metered_2016.csv: 245,952 rows
  ✓ hrl_load_metered_2017.csv: 250,417 rows
  ✓ hrl_load_metered_2018.csv: 254,784 rows
  ✓ hrl_load_metered_2019.csv: 262,800 rows
  ✓ hrl_load_metered_2020.csv: 263,520 rows
  ✓ hrl_load_metered_2021.csv: 262,800 rows
  ✓ hrl_load_metered_2022.csv: 262,800 rows
  ✓ hrl_load_metered_2023.csv: 262,800 rows
  ✓ hrl_load_metered_2024.csv: 263,520 rows
  ✓ hrl_load_metered_2025.csv: 218,130 rows
  ✓ hrl_load_metered_update.csv: 15 rows

PJM Data Summary:
Total rows: 3,367,288
Total columns: 8
Date range: 2012-01-01 00:00:00 to 2025-11-13 23:00:00
Load areas: 34

Columns: ['datetime_beginning_utc', 'datetime_beginning_ept', 'nerc_region', 'mkt_region', 'zone', 'load_area', 'mw', 'is_verified']

First 5 rows:


,datetime_beginning_utc,datetime_beginning_ept,nerc_region,mkt_region,zone,load_area,mw,is_verified
0,1/1/2012 5:00:00 AM,2012-01-01,RFC,MIDATL,BC,BC,3117.523,True
1,1/1/2012 5:00:00 AM,2012-01-01,RFC,MIDATL,CNCT,AE,1017.566,True
2,1/1/2012 5:00:00 AM,2012-01-01,RFC,MIDATL,CNCT,DPL,1745.034,True
3,1/1/2012 5:00:00 AM,2012-01-01,RFC,MIDATL,GPU,JC,2171.400,True
4,1/1/2012 5:00:00 AM,2012-01-01,RFC,MIDATL,GPU,ME,1377.857,True



✓ PJM data loaded successfully


## Basic Time Features

In [59]:
print("Creating basic time features...")

# Time components
full_df['year'] = full_df['datetime_beginning_ept'].dt.year
full_df['hour'] = full_df['datetime_beginning_ept'].dt.hour
full_df['day'] = full_df['datetime_beginning_ept'].dt.day
full_df['day_of_week'] = full_df['datetime_beginning_ept'].dt.dayofweek  # 0=Monday, 6=Sunday
full_df['day_of_year'] = full_df['datetime_beginning_ept'].dt.dayofyear
full_df['week_of_year'] = full_df['datetime_beginning_ept'].dt.isocalendar().week
full_df['month'] = full_df['datetime_beginning_ept'].dt.month
full_df['quarter'] = full_df['datetime_beginning_ept'].dt.quarter

# Weekend indicator
full_df['is_weekend'] = (full_df['day_of_week'] >= 5).astype(int)

# Time of day categories
full_df['time_of_day'] = pd.cut(full_df['hour'], 
                                  bins=[0, 6, 12, 18, 24],
                                  labels=['night', 'morning', 'afternoon', 'evening'],
                                  include_lowest=True)

# Cyclical encoding for hour and month (for models that benefit from it)
full_df['hour_sin'] = np.sin(2 * np.pi * full_df['hour'] / 24)
full_df['hour_cos'] = np.cos(2 * np.pi * full_df['hour'] / 24)
full_df['month_sin'] = np.sin(2 * np.pi * full_df['month'] / 12)
full_df['month_cos'] = np.cos(2 * np.pi * full_df['month'] / 12)
full_df['day_of_week_sin'] = np.sin(2 * np.pi * full_df['day_of_week'] / 7)
full_df['day_of_week_cos'] = np.cos(2 * np.pi * full_df['day_of_week'] / 7)

print("Basic time features created")
print(f"New columns: {list(full_df.columns[-15:])}")

Creating basic time features...
Basic time features created
New columns: ['hour', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'month', 'quarter', 'is_weekend', 'time_of_day', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']


## Black Friday and Holiday Features

In [60]:
print("Creating Black Friday and holiday features...")

# Black Friday dates
black_fridays = {year: get_black_friday(year) for year in range(2016, 2025)}

# Add Black Friday date for each row
full_df['black_friday'] = full_df['year'].map(black_fridays)

# Days from Black Friday
full_df['days_from_bf'] = (full_df['datetime_beginning_ept'] - full_df['black_friday']).dt.days

# Is Black Friday week (7 days before to Black Friday)
full_df['is_bf_week'] = ((full_df['days_from_bf'] >= -7) & (full_df['days_from_bf'] <= 0)).astype(int)

# Black Friday (days_from_bf == 0)
full_df['is_blackfriday'] = (full_df['days_from_bf'] == 0).astype(int)

# Thanksgiving (day before Black Friday)
full_df['is_thanksgiving'] = (full_df['days_from_bf'] == -1).astype(int)

# Holiday indicator
def is_holiday(row):
    holidays = get_us_holidays(row['year'])
    return int(row['datetime_beginning_ept'].date() in [h.date() for h in holidays])

full_df['is_holiday'] = full_df.apply(is_holiday, axis=1)

print("Black Friday and holiday features created")
print(f"Black Friday dates: {black_fridays}")
print(f"Added is_blackfriday and is_thanksgiving binary features")

Creating Black Friday and holiday features...
Black Friday and holiday features created
Black Friday dates: {2016: datetime.datetime(2016, 11, 25, 0, 0), 2017: datetime.datetime(2017, 11, 24, 0, 0), 2018: datetime.datetime(2018, 11, 23, 0, 0), 2019: datetime.datetime(2019, 11, 29, 0, 0), 2020: datetime.datetime(2020, 11, 27, 0, 0), 2021: datetime.datetime(2021, 11, 26, 0, 0), 2022: datetime.datetime(2022, 11, 25, 0, 0), 2023: datetime.datetime(2023, 11, 24, 0, 0), 2024: datetime.datetime(2024, 11, 29, 0, 0)}
Added is_blackfriday and is_thanksgiving binary features


## Lag Features (Historical Values)

In [61]:
print("Creating lag features...")
print("This may take several minutes...")

# Sort by load_area and datetime for proper lag calculation
full_df = full_df.sort_values(['load_area', 'datetime_beginning_ept']).reset_index(drop=True)

# Define lag periods (in hours)
# Include all lags from 1h to 1year for horizon-aware prediction
# During training, we'll mask lags based on prediction horizon
lag_hours = [1, 2, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168, 192, 216, 336, 720, 365*24, 365*24*2, 365*24*3, 365*24*4]  
# 1h, 2h, 3h, 6h, 12h, 1day, 2days, 3days, 4days, 5days, 6days, 1week, 8days, 9days, 2weeks, 30days, 1year, 2years, 3years, 4years

for lag in lag_hours:
    lag_name = f"{lag}h"
    if lag == 24:
        lag_name = "24h (1day)"
    elif lag == 48:
        lag_name = "48h (2days)"
    elif lag == 72:
        lag_name = "72h (3days)"
    elif lag == 96:
        lag_name = "96h (4days)"
    elif lag == 120:
        lag_name = "120h (5days)"
    elif lag == 144:
        lag_name = "144h (6days)"
    elif lag == 168:
        lag_name = "168h (1week)"
    elif lag == 192:
        lag_name = "192h (8days)"
    elif lag == 216:
        lag_name = "216h (9days)"
    elif lag == 336:
        lag_name = "336h (2weeks)"
    elif lag == 720:
        lag_name = "720h (30days)"
    elif lag == 365*24:
        lag_name = "8760h (1year)"
    elif lag == 365*24*2:
        lag_name = "17520h (2years)"
    elif lag == 365*24*3:
        lag_name = "26280h (3years)"
    elif lag == 365*24*4:
        lag_name = "35040h (4years)"
    
    print(f"  Creating lag_{lag_name}...")
    full_df[f'mw_lag_{lag}h'] = full_df.groupby('load_area')['mw'].shift(lag)

print("Lag features created")
print(f"Note: All lags included for horizon-aware prediction")
print(f"      Lags will be masked based on prediction horizon during training")
print(f"      Removed duplicate: mw_lag_1week_same_hour")

Creating lag features...
This may take several minutes...
  Creating lag_1h...
  Creating lag_2h...
  Creating lag_3h...
  Creating lag_6h...
  Creating lag_12h...
  Creating lag_24h (1day)...
  Creating lag_48h (2days)...
  Creating lag_72h (3days)...
  Creating lag_96h (4days)...
  Creating lag_120h (5days)...
  Creating lag_144h (6days)...
  Creating lag_168h (1week)...
  Creating lag_192h (8days)...
  Creating lag_216h (9days)...
  Creating lag_336h (2weeks)...
  Creating lag_720h (30days)...
  Creating lag_8760h (1year)...
  Creating lag_17520h (2years)...
  Creating lag_26280h (3years)...
  Creating lag_35040h (4years)...
Lag features created
Note: All lags included for horizon-aware prediction
      Lags will be masked based on prediction horizon during training
      Removed duplicate: mw_lag_1week_same_hour


## Black Friday-Based Lag Features

Create lags based on Black Friday relative position (BF-9, BF-8, ..., BF)

For example: This year's BF-9 → Last year's BF-9 (same relative position, different calendar date)

In [62]:
print("Creating Black Friday-based lag features...")
print("This matches same relative BF position across years")
print("Example: This year BF-9 -> Last year BF-9\n")

# Create a lookup dictionary for each (load_area, days_from_bf, hour, year)
print("  Building BF-based lookup table...")
bf_lookup = full_df.set_index(['load_area', 'days_from_bf', 'hour', 'year'])['mw'].to_dict()

# BF-based lags: 1 year, 2 years, 3 years, 4 years ago
bf_lag_years = [1, 2, 3, 4]

for lag_yr in bf_lag_years:
    print(f"  Creating bf_lag_{lag_yr}yr (BF position {lag_yr} year(s) ago)...")
    
    def get_bf_lag(row):
        # Look up same (area, BF relative day, hour) but lag_yr years ago
        key = (row['load_area'], row['days_from_bf'], row['hour'], row['year'] - lag_yr)
        return bf_lookup.get(key, np.nan)
    
    full_df[f'mw_bf_lag_{lag_yr}yr'] = full_df.apply(get_bf_lag, axis=1)

print("\nBF-based lag features created")
print(f"Features: mw_bf_lag_1yr, mw_bf_lag_2yr, mw_bf_lag_3yr, mw_bf_lag_4yr")
print(f"\nExample: For 2024 BF-9 (Nov 20, 10am):")
print(f"  bf_lag_1yr = 2023 BF-9 (Nov 15, 10am) value")
print(f"  bf_lag_2yr = 2022 BF-9 (Nov 16, 10am) value")
print(f"  (Same relative position, different calendar dates)")

Creating Black Friday-based lag features...
This matches same relative BF position across years
Example: This year BF-9 -> Last year BF-9

  Building BF-based lookup table...
  Creating bf_lag_1yr (BF position 1 year(s) ago)...
  Creating bf_lag_2yr (BF position 2 year(s) ago)...
  Creating bf_lag_3yr (BF position 3 year(s) ago)...
  Creating bf_lag_4yr (BF position 4 year(s) ago)...

BF-based lag features created
Features: mw_bf_lag_1yr, mw_bf_lag_2yr, mw_bf_lag_3yr, mw_bf_lag_4yr

Example: For 2024 BF-9 (Nov 20, 10am):
  bf_lag_1yr = 2023 BF-9 (Nov 15, 10am) value
  bf_lag_2yr = 2022 BF-9 (Nov 16, 10am) value
  (Same relative position, different calendar dates)


## Rolling Statistics

In [63]:
print("Creating rolling statistics...")
print("This may take several minutes...")

# Define rolling windows (in hours)
windows = [24, 168, 720]  # 1 day, 1 week, 30 days

for window in windows:
    window_name = f"{window}h"
    if window == 24:
        window_name = "24h"
    elif window == 168:
        window_name = "7d"
    elif window == 720:
        window_name = "30d"
    
    print(f"  Creating rolling stats for {window_name}...")
    
    # Rolling mean
    full_df[f'mw_rolling_mean_{window_name}'] = full_df.groupby('load_area')['mw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )
    
    # Rolling std
    full_df[f'mw_rolling_std_{window_name}'] = full_df.groupby('load_area')['mw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).std()
    )
    
    # Rolling min
    full_df[f'mw_rolling_min_{window_name}'] = full_df.groupby('load_area')['mw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).min()
    )
    
    # Rolling max
    full_df[f'mw_rolling_max_{window_name}'] = full_df.groupby('load_area')['mw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).max()
    )

print("Rolling statistics created")

Creating rolling statistics...
This may take several minutes...
  Creating rolling stats for 24h...
  Creating rolling stats for 7d...
  Creating rolling stats for 30d...
Rolling statistics created


## Data Summary

In [64]:
print("\n=== Feature Engineering Summary ===")
print(f"Total rows: {len(full_df):,}")
print(f"Total features: {len(full_df.columns)}")
print(f"\nFeature columns:")
for col in full_df.columns:
    print(f"  - {col}")

print(f"\nMissing values:")
missing = full_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print(missing)
else:
    print("  No missing values")

print(f"\nData types:")
print(full_df.dtypes.value_counts())


=== Feature Engineering Summary ===
Total rows: 3,367,288
Total features: 66

Feature columns:
  - datetime_beginning_utc
  - datetime_beginning_ept
  - nerc_region
  - mkt_region
  - zone
  - load_area
  - mw
  - is_verified
  - year
  - hour
  - day
  - day_of_week
  - day_of_year
  - week_of_year
  - month
  - quarter
  - is_weekend
  - time_of_day
  - hour_sin
  - hour_cos
  - month_sin
  - month_cos
  - day_of_week_sin
  - day_of_week_cos
  - black_friday
  - days_from_bf
  - is_bf_week
  - is_blackfriday
  - is_thanksgiving
  - is_holiday
  - mw_lag_1h
  - mw_lag_2h
  - mw_lag_3h
  - mw_lag_6h
  - mw_lag_12h
  - mw_lag_24h
  - mw_lag_48h
  - mw_lag_72h
  - mw_lag_96h
  - mw_lag_120h
  - mw_lag_144h
  - mw_lag_168h
  - mw_lag_192h
  - mw_lag_216h
  - mw_lag_336h
  - mw_lag_720h
  - mw_lag_8760h
  - mw_lag_17520h
  - mw_lag_26280h
  - mw_lag_35040h
  - mw_bf_lag_1yr
  - mw_bf_lag_2yr
  - mw_bf_lag_3yr
  - mw_bf_lag_4yr
  - mw_rolling_mean_24h
  - mw_rolling_std_24h
  - mw_rolling_

## Sample Data Check

In [65]:
# Show sample for one area
sample_area = 'AEPIMP'
sample_data = full_df[full_df['load_area'] == sample_area].head(50)

print(f"\n=== Sample Data for {sample_area} ===")
print(sample_data[[
    'datetime_beginning_ept', 'mw', 'hour', 'day_of_week', 'is_weekend',
    'days_from_bf', 'is_bf_week', 'mw_lag_48h', 'mw_lag_168h', 'mw_rolling_mean_24h'
]].head(20))


=== Sample Data for AEPIMP ===
       datetime_beginning_ept        mw  hour  day_of_week  is_weekend  \
242494    2015-06-01 00:00:00  2415.634     0            0           0   
242495    2015-06-01 01:00:00  2351.179     1            0           0   
242496    2015-06-01 02:00:00  2312.083     2            0           0   
242497    2015-06-01 03:00:00  2268.070     3            0           0   
242498    2015-06-01 04:00:00  2410.219     4            0           0   
242499    2015-06-01 05:00:00  2545.226     5            0           0   
242500    2015-06-01 06:00:00  2752.902     6            0           0   
242501    2015-06-01 07:00:00  3000.023     7            0           0   
242502    2015-06-01 08:00:00  3087.455     8            0           0   
242503    2015-06-01 09:00:00  3181.535     9            0           0   
242504    2015-06-01 10:00:00  3196.300    10            0           0   
242505    2015-06-01 11:00:00  3196.089    11            0           0   
242506

## Save Engineered Features

In [66]:
# Save full engineered dataset
output_file = '../dataset/preprocessed/full_year_features.csv'
print(f"Saving to {output_file}...")
full_df.to_csv(output_file, index=False)

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"\nSaved successfully!")
print(f"File: {output_file}")
print(f"Size: {file_size_mb:.2f} MB")
print(f"Rows: {len(full_df):,}")
print(f"Columns: {len(full_df.columns)}")

Saving to ../dataset/preprocessed/full_year_features.csv...

Saved successfully!
File: ../dataset/preprocessed/full_year_features.csv
Size: 1848.13 MB
Rows: 3,367,288
Columns: 66


## Save Parquet Format (Faster Loading)

In [67]:
# Parquet is more efficient for large datasets
parquet_file = '../dataset/preprocessed/full_year_features.parquet'
print(f"Saving to {parquet_file}...")
full_df.to_parquet(parquet_file, index=False)

parquet_size_mb = os.path.getsize(parquet_file) / (1024 * 1024)
print(f"\nParquet file saved!")
print(f"File: {parquet_file}")
print(f"Size: {parquet_size_mb:.2f} MB (vs CSV: {file_size_mb:.2f} MB)")
print(f"Compression ratio: {file_size_mb / parquet_size_mb:.2f}x")

Saving to ../dataset/preprocessed/full_year_features.parquet...

Parquet file saved!
File: ../dataset/preprocessed/full_year_features.parquet
Size: 599.34 MB (vs CSV: 1848.13 MB)
Compression ratio: 3.08x


## Feature Importance Preview

In [68]:
# Show correlation of features with target (mw)
numeric_cols = full_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('mw')  # Remove target

correlations = full_df[numeric_cols + ['mw']].corr()['mw'].abs().sort_values(ascending=False)

print("\n=== Top 20 Features by Correlation with MW ===")
print(correlations.head(21))  # 20 + target itself


=== Top 20 Features by Correlation with MW ===
mw                     1.000000
mw_lag_1h              0.999192
mw_lag_2h              0.996987
mw_lag_24h             0.996450
mw_lag_3h              0.993789
mw_lag_48h             0.992221
mw_lag_168h            0.991282
mw_bf_lag_3yr          0.991191
mw_rolling_mean_24h    0.991121
mw_bf_lag_2yr          0.991107
mw_bf_lag_4yr          0.990967
mw_bf_lag_1yr          0.990941
mw_rolling_max_24h     0.990266
mw_lag_72h             0.990180
mw_lag_144h            0.990160
mw_lag_336h            0.989999
mw_lag_192h            0.989661
mw_lag_8760h           0.989365
mw_lag_96h             0.989199
mw_lag_120h            0.988944
mw_rolling_min_24h     0.988181
Name: mw, dtype: float64
